# Cross-Dataset Experiment — octc8 & octid
Evaluate the trained OCTNet on shared classes from new datasets.
Then run the full ablation study on octc8 shared classes.

**Shared classes with your model (CNV, DME, DRUSEN, NORMAL):**
- octc8 → CNV, DME, DRUSEN, NORMAL (all 4 present)
- octid → NORMAL only


## 0. Setup — paste your full cell 0 here
All imports, config, class definitions, and model loading from tool5 notebook.

In [ ]:
# Paste your entire cell 0 from tool5_agent_orchestrator here
# (imports, OCTNet, GradCAMpp, SmallEncoder, CORPUS, tokenizer,
#  load eval_model, encoder, corpus_embs, tool definitions, TOOL_REGISTRY,
#  SYSTEM_PROMPT, run_agent, parse_tool_call, parse_final)
print('Cell 0 loaded.')


## 1. Helper — filtered ImageFolder
Loads only the classes your model knows about.

In [ ]:
import os
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
from pathlib import Path
from sklearn.metrics import accuracy_score, f1_score, classification_report
import numpy as np
from tqdm import tqdm

# Your model's classes — must match CKPT_PATH training
MODEL_CLASSES = ['CNV', 'DME', 'DRUSEN', 'NORMAL']


def load_shared_dataset(folder: str, model_classes: list, transform) -> datasets.ImageFolder:
    """
    Load an ImageFolder dataset but keep ONLY classes the model knows.
    Remaps class indices so they match MODEL_CLASSES order.
    """
    full_ds = datasets.ImageFolder(folder, transform=transform)
    available = set(full_ds.classes)
    shared    = [c for c in model_classes if c in available]
    missing   = [c for c in model_classes if c not in available]

    print(f'Folder       : {folder}')
    print(f'All classes  : {full_ds.classes}')
    print(f'Shared       : {shared}')
    print(f'Not in folder: {missing}')

    # Keep only samples whose class is in shared
    shared_set = set(shared)
    indices = [i for i, (_, label_idx) in enumerate(full_ds.samples)
               if full_ds.classes[label_idx] in shared_set]

    # Build a filtered subset
    subset = Subset(full_ds, indices)

    # Store metadata for easy access
    subset.shared_classes    = shared
    subset.full_ds           = full_ds
    subset.model_classes     = model_classes

    # Build a mapping: original label index -> model class index
    subset.label_remap = {
        full_ds.class_to_idx[c]: model_classes.index(c)
        for c in shared
    }

    print(f'Kept samples : {len(indices)}')
    return subset


def get_sample_list(subset) -> list:
    """Return list of (path, remapped_label) for a filtered subset."""
    samples = []
    for idx in subset.indices:
        path, orig_label = subset.full_ds.samples[idx]
        remapped = subset.label_remap[orig_label]
        samples.append((path, remapped))
    return samples


print('Helper functions defined.')


## 2. Load octc8 test set (shared classes)

In [ ]:
OCTC8_TEST = 'octc8/test'   # ← change path if needed

octc8_subset = load_shared_dataset(OCTC8_TEST, MODEL_CLASSES, val_tf)
octc8_samples = get_sample_list(octc8_subset)
print(f'\noctc8 test samples (shared classes): {len(octc8_samples)}')


## 3. Load octid (shared classes)
octid has no train/test split — use all images for evaluation.

In [ ]:
OCTID_ROOT = 'octid'   # ← change path if needed

octid_subset = load_shared_dataset(OCTID_ROOT, MODEL_CLASSES, val_tf)
octid_samples = get_sample_list(octid_subset)
print(f'\noctid samples (shared classes): {len(octid_samples)}')


## 4. Classifier-only baseline on both datasets
Before running the agent, check raw OCTNet accuracy on these new datasets.
This measures how well your from-scratch model generalizes.

In [ ]:
from PIL import Image as PILImage


def evaluate_classifier_only(samples: list, dataset_name: str) -> dict:
    """Run OCTNet directly (no agent) and report accuracy + F1."""
    preds, trues = [], []
    for path, true_label in tqdm(samples, desc=f'Classifier eval [{dataset_name}]'):
        result = tool_coarse_classify(path)
        pred_label = MODEL_CLASSES.index(result['predicted_class'])
        preds.append(pred_label)
        trues.append(true_label)

    acc = accuracy_score(trues, preds)
    f1  = f1_score(trues, preds, average='macro',
                   labels=list(range(len(MODEL_CLASSES))), zero_division=0)
    print(f'\n=== {dataset_name} — Classifier Only ===')
    print(f'Accuracy : {acc*100:.2f}%')
    print(f'Macro F1 : {f1*100:.2f}%')
    print(classification_report(trues, preds, target_names=MODEL_CLASSES,
                                 zero_division=0, digits=4))
    return {'accuracy': acc, 'macro_f1': f1, 'preds': preds, 'trues': trues}


octc8_baseline = evaluate_classifier_only(octc8_samples, 'octc8')
octid_baseline = evaluate_classifier_only(octid_samples, 'octid')


## 5. Ablation study on octc8
octc8 has all 4 of your model's classes so it's the right dataset for ablation.
Uses up to 40 images per class (160 total) to keep API calls manageable.

In [ ]:
from collections import defaultdict


def ablation_run(image_path: str, disabled_tools: list) -> str:
    originals = {}
    for t in disabled_tools:
        originals[t] = TOOL_REGISTRY[t]
        TOOL_REGISTRY[t] = lambda **_: {'error': 'tool disabled for ablation'}
    result = run_agent(image_path, verbose=False)
    for t, fn in originals.items():
        TOOL_REGISTRY[t] = fn
    return result.get('finding', 'UNKNOWN')


# Build balanced subset — up to 40 images per shared class
per_class = defaultdict(list)
for path, label in octc8_samples:
    if len(per_class[label]) < 40:
        per_class[label].append((path, label))

ablation_subset = [item for items in per_class.values() for item in items]
print(f'Ablation subset: {len(ablation_subset)} images')
for label_idx, items in per_class.items():
    print(f'  {MODEL_CLASSES[label_idx]}: {len(items)} images')


In [ ]:
import time

ABLATION_CONDITIONS = {
    'Full pipeline':            [],
    'No zoom (-Tool3)':         ['zoom_reanalyze'],
    'No retrieval (-Tool4)':    ['retrieve_knowledge'],
    'No localization (-Tool2)': ['localize'],
    'Classifier only (-2,3,4)': ['localize', 'zoom_reanalyze', 'retrieve_knowledge'],
}

ablation_results_octc8 = {}

for condition, disabled in ABLATION_CONDITIONS.items():
    preds, trues = [], []
    for path, label in tqdm(ablation_subset, desc=condition):
        pred_name = ablation_run(path, disabled)
        pred_idx  = MODEL_CLASSES.index(pred_name) if pred_name in MODEL_CLASSES else -1
        preds.append(pred_idx)
        trues.append(label)
        time.sleep(0.5)   # small delay to avoid rate limits

    acc = accuracy_score(trues, preds)
    f1  = f1_score(trues, preds, average='macro',
                   labels=list(range(len(MODEL_CLASSES))), zero_division=0)
    ablation_results_octc8[condition] = {'accuracy': acc, 'macro_f1': f1}
    print(f'{condition:35s} | Acc {acc*100:.1f}% | F1 {f1*100:.1f}%')


## 6. Ablation on uncertain cases only
Filter to images where classifier confidence < 85%.
This is where agent tools are most likely to show measurable contribution.

In [ ]:
print('Finding uncertain cases (confidence < 85%)...')
uncertain = []
for path, label in tqdm(octc8_samples, desc='Scanning confidence'):
    result = tool_coarse_classify(path)
    if result['confidence'] < 0.85:
        uncertain.append((path, label, result['confidence']))

print(f'Uncertain cases: {len(uncertain)} / {len(octc8_samples)} ({len(uncertain)/len(octc8_samples)*100:.1f}%)')
for path, label, conf in uncertain[:10]:
    print(f'  {MODEL_CLASSES[label]} | conf={conf:.3f} | {Path(path).name}')


In [ ]:
if len(uncertain) == 0:
    print('No uncertain cases found — classifier is highly confident on octc8.')
    print('This is a strong generalization result but limits ablation signal.')
else:
    uncertain_samples = [(p, l) for p, l, _ in uncertain]
    ablation_results_uncertain = {}

    for condition, disabled in ABLATION_CONDITIONS.items():
        preds, trues = [], []
        for path, label in tqdm(uncertain_samples, desc=f'[uncertain] {condition}'):
            pred_name = ablation_run(path, disabled)
            pred_idx  = MODEL_CLASSES.index(pred_name) if pred_name in MODEL_CLASSES else -1
            preds.append(pred_idx)
            trues.append(label)
            time.sleep(0.5)

        acc = accuracy_score(trues, preds)
        f1  = f1_score(trues, preds, average='macro',
                       labels=list(range(len(MODEL_CLASSES))), zero_division=0)
        ablation_results_uncertain[condition] = {'accuracy': acc, 'macro_f1': f1}
        print(f'{condition:35s} | Acc {acc*100:.1f}% | F1 {f1*100:.1f}%')


## 7. Summary table — all results

In [ ]:
import pandas as pd

print('\n=== CLASSIFIER BASELINE (no agent) ===')
baseline_df = pd.DataFrame([
    {'Dataset': 'OCT2017 (original)', 'Accuracy (%)': '99.4', 'Macro F1 (%)': '99.7'},
    {'Dataset': 'octc8 shared classes', 
     'Accuracy (%)': f"{octc8_baseline['accuracy']*100:.1f}",
     'Macro F1 (%)': f"{octc8_baseline['macro_f1']*100:.1f}"},
    {'Dataset': 'octid shared classes',
     'Accuracy (%)': f"{octid_baseline['accuracy']*100:.1f}",
     'Macro F1 (%)': f"{octid_baseline['macro_f1']*100:.1f}"},
])
print(baseline_df.to_string(index=False))

print('\n=== ABLATION — octc8 shared classes ===')
ablation_df = pd.DataFrame([
    {'Condition': cond,
     'Accuracy (%)': f"{v['accuracy']*100:.1f}",
     'Macro F1 (%)': f"{v['macro_f1']*100:.1f}"}
    for cond, v in ablation_results_octc8.items()
])
print(ablation_df.to_string(index=False))


## 8. Confusion matrices

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix


def plot_cm(trues, preds, class_names, title, filename):
    cm = confusion_matrix(trues, preds, labels=list(range(len(class_names))))
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.title(title)
    plt.ylabel('True'); plt.xlabel('Predicted')
    plt.tight_layout()
    plt.savefig(filename, dpi=150)
    plt.show()


plot_cm(octc8_baseline['trues'], octc8_baseline['preds'],
        MODEL_CLASSES, 'OCTNet on octc8 (shared classes)', 'cm_octc8.png')

plot_cm(octid_baseline['trues'], octid_baseline['preds'],
        MODEL_CLASSES, 'OCTNet on octid (shared classes)', 'cm_octid.png')
